In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/PlasticEnz.db")

tables = pd.read_sql("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

print(tables)

              name
0        Databases
1          Enzymes
2        Main_copy
3   Microorganisms
4          Plastic
5         Reaction
6        Sequences
7            Study
8  sqlite_sequence


In [2]:
for table in tables["name"]:
    print(f"\n--- {table} ---")

    info = pd.read_sql(
        f'PRAGMA table_info("{table}")',
        conn
    )

    print(info[["name", "type"]].to_string(index=False))


--- Databases ---
     name    type
EC_number DECIMAL
 Uni_prot    TEXT
     NCBI  STRING
    Db_Pk INTEGER
    Genus        
  Species        
   Strain        

--- Enzymes ---
            name    type
        Location        
            Gene        
            Type        
     Superfamily        
          Operon        
    Gene_Cluster        
Catalytic domain        
      Enzymes_id INTEGER

--- Main_copy ---
              name    type
             Genus    TEXT
           Species    TEXT
            Strain    TEXT
            Enzyme    TEXT
Optimal_conditions    TEXT
           Main_Pk INTEGER
       Database_id     INT
      Sequences_id     INT
          Study_id     INT
        Enzymes_id     INT
  Microorganism_id     INT
       Reaction_id     INT
        Plastic_id     INT

--- Microorganisms ---
            name    type
MARKER GENE TYPE        
          SOURCE        
           MO_pk INTEGER
            TYPE        
           Genus    TEXT
         Species    TEXT

In [3]:
print("\n--- Main_copy sample ---")

main = pd.read_sql("""
    SELECT *
    FROM Main_copy
    LIMIT 10
""", conn)

print(main.to_string(index=False))


--- Main_copy sample ---
       Genus        Species        Strain            Enzyme                                                                                                                                                                                                                                                                                                                                                                                                     Optimal_conditions  Main_Pk  Database_id  Sequences_id  Study_id  Enzymes_id  Microorganism_id  Reaction_id  Plastic_id
Streptomyces  hygroscopicus ascomyceticus PHB depolymerase  he enzyme was found to be a monomer with a molecular mass of 48.4 kDa, and displayed highest activity at 45°C and pH 6, thus being the first PHB depolymerase from a gram-positive bacterium presenting an acidic pH optimum.The amino acids comprising the catalytic triad, Ser131-Asp209-His269, were identified by multiple sequence alignment, chemica

In [4]:
print("\n--- Number of rows ---")

print(
    pd.read_sql("""
        SELECT COUNT(*) AS rows
        FROM Main_copy
    """, conn)
)


--- Number of rows ---
   rows
0   279


In [5]:
print("\n--- Plastic counts ---")

print(
    pd.read_sql("""
        SELECT Plastic_id, COUNT(*) AS count
        FROM Main_copy
        GROUP BY Plastic_id
        ORDER BY count DESC
    """, conn)
)


--- Plastic counts ---
    Plastic_id  count
0          1.0     74
1          3.0     55
2          5.0     46
3          9.0     23
4         10.0     18
5          6.0     17
6          4.0     12
7         11.0      9
8          7.0      9
9          NaN      7
10        13.0      4
11         2.0      3
12        12.0      1
13         8.0      1


In [6]:
query = """
SELECT
    m.Main_Pk,
    m.Enzyme,
    s.SEQUENCE,
    e.Gene,
    e.Type AS Enzyme_Type,
    e.Superfamily,
    e.Operon,
    e.Gene_Cluster,
    e."Catalytic domain" AS Catalytic_Domain,
    p.PLASTIC AS Plastic,
    p.TYPE AS Plastic_Type
FROM Main_copy m

LEFT JOIN Sequences s
    ON m.Sequences_id = s.Seq_Pk

LEFT JOIN Enzymes e
    ON m.Enzymes_id = e.Enzymes_id

LEFT JOIN Plastic p
    ON m.Plastic_id = p.Plastic_id
"""

df1 = pd.read_sql(query, conn)

print(df1.shape)
print(df1.head())

(279, 11)
   Main_Pk             Enzyme  \
0        1  PHB depolymerase    
1        2  PHB depolymerase    
2        3  PHB depolymerase    
3        4  PHB depolymerase    
4        5  PHB depolymerase    

                                            SEQUENCE         Gene  \
0  MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...  fkbU (PhaZ)   
1  MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...         PhaZ   
2  MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNF\n...         PhaZ   
3  MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...         PhaZ   
4  MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...         PhaZ   

              Enzyme_Type Superfamily      Operon Gene_Cluster  \
0  Acting on ester bonds   Hydrolases                    FK520   
1  Acting on ester bonds   Hydrolases  pha operon                
2  Acting on ester bonds   Hydrolases  pha operon                
3  Acting on ester bonds   Hydrolases  pha operon                
4  Acting on ester bonds   Hydrolases  pha ope

In [7]:
print("\nMissing values:")
print(df1.isna().sum())

print("\nPlastic classes:")
print(df1["Plastic"].value_counts(dropna=False))


Missing values:
Main_Pk              0
Enzyme               7
SEQUENCE             8
Gene                 7
Enzyme_Type          7
Superfamily          7
Operon              68
Gene_Cluster        71
Catalytic_Domain    68
Plastic              7
Plastic_Type         7
dtype: int64

Plastic classes:
Plastic
PET          74
PHB          55
PCL          46
PBSA         23
PBAT         18
n-alkanes    17
PLA          12
PHA           9
Nylon         9
NaN           7
NR            4
PE            3
MHET          1
PHO           1
Name: count, dtype: int64


In [8]:
print("\nSequence availability:")
print(df1["SEQUENCE"].notna().value_counts())


Sequence availability:
SEQUENCE
True     271
False      8
Name: count, dtype: int64


In [9]:
print("\nSequence availability:")
print(df1["SEQUENCE"].notna().value_counts())


Sequence availability:
SEQUENCE
True     271
False      8
Name: count, dtype: int64


In [10]:
sequence_plastic = (
    df1.dropna(subset=["SEQUENCE", "Plastic"])
       .groupby("SEQUENCE")["Plastic"]
       .nunique()
)

print(sequence_plastic.value_counts())

Plastic
1    205
2     21
3      4
5      1
Name: count, dtype: int64
